In [1]:

"""
ZRA VAT FRAUD DETECTION SYSTEM
Feature Engineering Pipeline - ROBUST VERSION
=================================
Auto-adapts to your dataset column names
"""

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("ZRA VAT FRAUD DETECTION - FEATURE ENGINEERING PIPELINE")
print("="*70)


ZRA VAT FRAUD DETECTION - FEATURE ENGINEERING PIPELINE


In [2]:
print("\n[STEP 1] Loading and inspecting datasets...")

try:
    claims = pd.read_csv(r"C:\Users\dell\Desktop\ZRA DATASETS\zra_vatClaims.csv")
    taxpayers = pd.read_csv(r"C:\Users\dell\Desktop\ZRA DATASETS\zra_taxpayerMaster.csv")
    purchases = pd.read_csv(r"C:\Users\dell\Desktop\ZRA DATASETS\zra_purchaseTransactions.csv")
    sales = pd.read_csv(r"C:\Users\dell\Desktop\ZRA DATASETS\zra_salesTransactions.csv")
    customs = pd.read_csv(r"C:\Users\dell\Desktop\ZRA DATASETS\zra_customsData.csv")
    compliance = pd.read_csv(r"C:\Users\dell\Desktop\ZRA DATASETS\zra_complianceHistory.csv")
    pacra = pd.read_csv(r"C:\Users\dell\Desktop\ZRA DATASETS\zra_thirdPartyVerification.csv")
    labels = pd.read_csv(r"C:\Users\dell\Desktop\ZRA DATASETS\zra_auditOutcomes.csv")
    
    print(f"✓ Claims: {len(claims):,} records")
    print(f"✓ Taxpayers: {len(taxpayers):,} records")
    print(f"✓ Purchases: {len(purchases):,} records")
    print(f"✓ Sales: {len(sales):,} records")
    print(f"✓ Customs: {len(customs):,} records")
    print(f"✓ Compliance: {len(compliance):,} records")
    print(f"✓ PACRA: {len(pacra):,} records")
    print(f"✓ Labels: {len(labels):,} records")
    
except FileNotFoundError as e:
    print(f"\n❌ ERROR: Could not find file: {e.filename}")
    print("\nMake sure all CSV files are in the same directory as this script:")
    print("  - zra_vatClaims.csv")
    print("  - zra_taxpayerMaster.csv")
    print("  - zra_purchaseTransactions.csv")
    print("  - zra_salesTransactions.csv")
    print("  - zra_customsData.csv")
    print("  - zra_complianceHistory.csv")
    print("  - zra_thirdPartyVerification.csv")
    print("  - zra_auditOutcomes.csv")
    exit()

# Print column names for debugging
print("\n[DEBUG] Column names in each dataset:")
print(f"Claims columns: {claims.columns.tolist()}")
print(f"Taxpayers columns: {taxpayers.columns.tolist()}")


[STEP 1] Loading and inspecting datasets...
✓ Claims: 1,000 records
✓ Taxpayers: 1,000 records
✓ Purchases: 10,596 records
✓ Sales: 14,751 records
✓ Customs: 971 records
✓ Compliance: 997 records
✓ PACRA: 1,000 records
✓ Labels: 1,000 records

[DEBUG] Column names in each dataset:
Claims columns: ['claim_id', 'taxpayer_id', 'claim_date', 'refund_period_start', 'refund_period_end', 'input_vat_claimed', 'output_vat_paid', 'net_refund_amount', 'claim_status', 'processing_days', 'submitted_by', 'supporting_docs_count', 'previous_refunds_count', 'sector']
Taxpayers columns: ['taxpayer_id', 'business_name', 'business_type', 'sector', 'registration_date', 'physical_address', 'postal_address', 'phone', 'email', 'directors', 'shareholders', 'tax_agent_tpin', 'annual_turnover', 'num_employees', 'bank_account_verified']


In [3]:
# ============================================================================
# STEP 2: BASIC CLAIM FEATURES (From VAT Claims Dataset)
# ============================================================================
print("\n[STEP 2] Engineering basic claim features...")

features = claims.copy()

# Feature 1: Refund Amount Intensity
features['refund_to_output_ratio'] = (
    features['net_refund_amount'] / (features['output_vat_paid'] + 1)
)

# Feature 2: Input to Output VAT Ratio (Classic fraud indicator)
features['input_output_vat_ratio'] = (
    features['input_vat_claimed'] / (features['output_vat_paid'] + 1)
)

# Feature 3: Processing Time Anomaly
features['processing_days_zscore'] = (
    (features['processing_days'] - features['processing_days'].mean()) / 
    (features['processing_days'].std() + 0.001)
)

# Feature 4: Claim Frequency (per taxpayer)
claim_counts = features.groupby('taxpayer_id').size().reset_index(name='total_claims_filed')
features = features.merge(claim_counts, on='taxpayer_id', how='left')

# Feature 5: First-time claimant flag
features['is_first_time_claimant'] = (features['previous_refunds_count'] == 0).astype(int)

# Feature 6: High-value claim flag (top 10%)
refund_threshold = features['net_refund_amount'].quantile(0.90)
features['is_high_value_claim'] = (features['net_refund_amount'] > refund_threshold).astype(int)

# Feature 7: Tax agent submission flag
features['submitted_by_agent'] = (features['submitted_by'] == 'Tax Agent').astype(int)

# Feature 8: Few supporting documents
features['has_few_documents'] = (features['supporting_docs_count'] < 5).astype(int)

print(f"✓ Created 8 basic claim features")


[STEP 2] Engineering basic claim features...
✓ Created 8 basic claim features


In [4]:
# ============================================================================
# STEP 3: TAXPAYER PROFILE FEATURES (From Taxpayer Master)
# ============================================================================
print("\n[STEP 3] Engineering taxpayer profile features...")

# Convert dates - handle different possible column names
date_cols_to_convert = ['registration_date', 'claim_date']
for df, col in [(taxpayers, 'registration_date'), (features, 'claim_date')]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Merge taxpayer info
features = features.merge(taxpayers, on='taxpayer_id', how='left')

# Feature 9: Business Age (in years)
if 'registration_date' in features.columns and 'claim_date' in features.columns:
    features['business_age_years'] = (
        (features['claim_date'] - features['registration_date']).dt.days / 365.25
    )
    features['business_age_years'].fillna(5, inplace=True)  # Default to 5 years if missing
else:
    features['business_age_years'] = 5  # Default value

# Feature 10: Young business claiming
features['is_young_business'] = (features['business_age_years'] < 2).astype(int)

# Feature 11: Unverified address
if 'physical_address' in features.columns:
    features['address_not_verified'] = features['physical_address'].str.contains('Not Verified', na=False).astype(int)
elif 'address_verified' in features.columns:
    features['address_not_verified'] = (features['address_verified'] == 'No').astype(int)
else:
    features['address_not_verified'] = 0

# Feature 12: Bank not verified
if 'bank_account_verified' in features.columns:
    features['bank_not_verified'] = (features['bank_account_verified'] == 'No').astype(int)
else:
    features['bank_not_verified'] = 0

# Feature 13: Small business (few employees)
if 'num_employees' in features.columns:
    features['is_small_business'] = (features['num_employees'] < 10).astype(int)
else:
    features['is_small_business'] = 0

# Feature 14: Claim amount relative to turnover
if 'annual_turnover' in features.columns:
    features['claim_to_turnover_ratio'] = (
        features['net_refund_amount'] / (features['annual_turnover'] + 1)
    )
else:
    features['claim_to_turnover_ratio'] = 0

# Feature 15: High claim for business size (>20% of turnover)
features['claim_exceeds_business_size'] = (
    features['claim_to_turnover_ratio'] > 0.20
).astype(int)

# Feature 16: Few directors (single person control)
if 'directors' in features.columns:
    features['single_director'] = (features['directors'].astype(str) == '1').astype(int)
else:
    features['single_director'] = 0

# Feature 17: No tax agent (self-filing can be higher risk)
if 'tax_agent_tpin' in features.columns:
    features['no_tax_agent'] = (features['tax_agent_tpin'] == 'None').astype(int)
else:
    features['no_tax_agent'] = 0

# Feature 18: Sole proprietor flag (higher risk than Ltd)
if 'business_type' in features.columns:
    features['is_sole_proprietor'] = (
        features['business_type'] == 'Sole Proprietor'
    ).astype(int)
else:
    features['is_sole_proprietor'] = 0

print(f"✓ Created 10 taxpayer profile features")


[STEP 3] Engineering taxpayer profile features...
✓ Created 10 taxpayer profile features


In [5]:
# ============================================================================
# STEP 4: PURCHASE TRANSACTION FEATURES (Supplier Analysis)
# ============================================================================
print("\n[STEP 4] Engineering purchase transaction features...")

# Aggregate purchase metrics per claim
purchase_agg = purchases.groupby('claim_id').agg({
    'supplier_tpin': 'nunique',
    'vat_amount': ['sum', 'mean', 'std'],
    'transaction_id': 'count'
}).reset_index()

purchase_agg.columns = [
    'claim_id', 'num_unique_suppliers', 'total_input_vat', 
    'avg_purchase_vat', 'std_purchase_vat', 'num_purchase_transactions'
]

# Count unverified suppliers and payments
if 'supplier_verified' in purchases.columns:
    unverified_suppliers = purchases.groupby('claim_id')['supplier_verified'].apply(
        lambda x: (x == 'No').sum()
    ).reset_index(name='num_unverified_suppliers')
    purchase_agg = purchase_agg.merge(unverified_suppliers, on='claim_id', how='left')
else:
    purchase_agg['num_unverified_suppliers'] = 0

if 'payment_verified' in purchases.columns:
    unverified_payments = purchases.groupby('claim_id')['payment_verified'].apply(
        lambda x: (x == 'No').sum()
    ).reset_index(name='num_unverified_payments')
    purchase_agg = purchase_agg.merge(unverified_payments, on='claim_id', how='left')
else:
    purchase_agg['num_unverified_payments'] = 0

features = features.merge(purchase_agg, on='claim_id', how='left')

# Fill NaN values with 0
for col in ['num_unique_suppliers', 'num_purchase_transactions', 'num_unverified_suppliers', 'num_unverified_payments']:
    features[col].fillna(0, inplace=True)

# Feature 19: Low supplier diversity
features['low_supplier_diversity'] = (features['num_unique_suppliers'] <= 3).astype(int)

# Feature 20: Has unverified suppliers
features['has_unverified_suppliers'] = (features['num_unverified_suppliers'] > 0).astype(int)

# Feature 21: High proportion of unverified suppliers
features['unverified_supplier_ratio'] = (
    features['num_unverified_suppliers'] / (features['num_unique_suppliers'] + 1)
)

# Feature 22: Has unverified payments
features['has_unverified_payments'] = (features['num_unverified_payments'] > 0).astype(int)

# Feature 23: Few purchase transactions
features['has_few_purchases'] = (features['num_purchase_transactions'] < 5).astype(int)

# Feature 24: Suspicious purchase pattern
features['std_purchase_vat'].fillna(0, inplace=True)
features['avg_purchase_vat'].fillna(1, inplace=True)
features['purchase_amount_consistency'] = (
    features['std_purchase_vat'] / (features['avg_purchase_vat'] + 1)
)
features['suspicious_purchase_pattern'] = (
    features['purchase_amount_consistency'] < 0.3
).astype(int)

print(f"✓ Created 6 purchase transaction features")



[STEP 4] Engineering purchase transaction features...
✓ Created 6 purchase transaction features


In [6]:
# ============================================================================
# STEP 5: SALES TRANSACTION FEATURES (Export Analysis)
# ============================================================================
print("\n[STEP 5] Engineering sales transaction features...")

# Aggregate sales metrics per claim
sales_agg = sales.groupby('claim_id').agg({
    'gross_amount': ['sum', 'mean'],
    'customer_country': 'nunique',
    'transaction_id': 'count'
}).reset_index()

sales_agg.columns = [
    'claim_id', 'total_sales_amount', 'avg_sale_amount',
    'num_customer_countries', 'num_sales_transactions'
]

# Count export sales
if 'is_export' in sales.columns:
    export_counts = sales.groupby('claim_id')['is_export'].apply(
        lambda x: (x == 'Yes').sum()
    ).reset_index(name='num_export_sales')
    sales_agg = sales_agg.merge(export_counts, on='claim_id', how='left')
else:
    sales_agg['num_export_sales'] = 0

# Count unpaid sales and missing docs
for col, new_name in [('payment_received', 'num_unpaid_sales'), ('export_doc_reference', 'num_missing_export_docs')]:
    if col in sales.columns:
        if col == 'payment_received':
            counts = sales.groupby('claim_id')[col].apply(lambda x: (x == 'No').sum()).reset_index(name=new_name)
        else:
            counts = sales.groupby('claim_id')[col].apply(lambda x: (x == 'N/A').sum()).reset_index(name=new_name)
        sales_agg = sales_agg.merge(counts, on='claim_id', how='left')
    else:
        sales_agg[new_name] = 0

features = features.merge(sales_agg, on='claim_id', how='left')

# Fill NaN
for col in ['num_export_sales', 'num_sales_transactions', 'num_unpaid_sales', 'num_missing_export_docs']:
    features[col].fillna(0, inplace=True)

# Feature 25-30
features['has_export_sales'] = (features['num_export_sales'] > 0).astype(int)
features['export_sales_ratio'] = features['num_export_sales'] / (features['num_sales_transactions'] + 1)
features['is_export_heavy'] = (features['export_sales_ratio'] > 0.5).astype(int)
features['has_missing_export_docs'] = (features['num_missing_export_docs'] > 0).astype(int)
features['has_unpaid_sales'] = (features['num_unpaid_sales'] > 0).astype(int)
features['has_few_sales'] = (features['num_sales_transactions'] < 10).astype(int)
features['single_export_destination'] = (features['num_customer_countries'] == 1).astype(int)

print(f"✓ Created 6 sales transaction features")


[STEP 5] Engineering sales transaction features...
✓ Created 6 sales transaction features


In [7]:
# ============================================================================
# STEP 6: CUSTOMS DATA FEATURES (Cross-Verification)
# ============================================================================
print("\n[STEP 6] Engineering customs verification features...")

# Export customs data
customs_agg = customs[customs['declaration_type'] == 'Export'].groupby('taxpayer_id').agg({
    'customs_value': 'sum',
    'declaration_id': 'count'
}).reset_index()

customs_agg.columns = ['taxpayer_id', 'total_customs_export_value', 'num_customs_declarations']

# Count unverified
if 'verified' in customs.columns:
    unverified = customs[customs['declaration_type'] == 'Export'].groupby('taxpayer_id')['verified'].apply(
        lambda x: (x == 'No').sum()
    ).reset_index(name='num_unverified_customs')
    customs_agg = customs_agg.merge(unverified, on='taxpayer_id', how='left')
else:
    customs_agg['num_unverified_customs'] = 0

features = features.merge(customs_agg, on='taxpayer_id', how='left')
features['total_customs_export_value'].fillna(0, inplace=True)
features['num_customs_declarations'].fillna(0, inplace=True)
features['num_unverified_customs'].fillna(0, inplace=True)

# Feature 31-33
features['export_without_customs'] = (
    (features['has_export_sales'] == 1) & 
    (features['num_customs_declarations'] == 0)
).astype(int)

features['total_sales_amount'].fillna(0, inplace=True)
features['export_customs_value_diff'] = features['total_sales_amount'] - features['total_customs_export_value']
features['large_export_customs_mismatch'] = (features['export_customs_value_diff'].abs() > 100000).astype(int)
features['has_unverified_customs'] = (features['num_unverified_customs'] > 0).astype(int)

print(f"✓ Created 3 customs verification features")



[STEP 6] Engineering customs verification features...
✓ Created 3 customs verification features


In [8]:
# ============================================================================
# STEP 7: COMPLIANCE HISTORY FEATURES
# ============================================================================
print("\n[STEP 7] Engineering compliance history features...")

compliance_agg = compliance.groupby('taxpayer_id').agg({
    'audit_date': 'count'
}).reset_index()
compliance_agg.columns = ['taxpayer_id', 'num_audits']

# Add conditional aggregations
if 'audit_result' in compliance.columns:
    issues = compliance.groupby('taxpayer_id')['audit_result'].apply(
        lambda x: (x == 'Issues Found').sum()
    ).reset_index(name='num_audits_with_issues')
    compliance_agg = compliance_agg.merge(issues, on='taxpayer_id', how='left')
else:
    compliance_agg['num_audits_with_issues'] = 0

for col in ['penalties_imposed', 'tax_adjustments', 'compliance_rating']:
    if col in compliance.columns:
        agg_data = compliance.groupby('taxpayer_id')[col].agg('sum' if col != 'compliance_rating' else 'mean').reset_index()
        agg_data.columns = ['taxpayer_id', f'total_{col}' if col != 'compliance_rating' else f'avg_{col}']
        compliance_agg = compliance_agg.merge(agg_data, on='taxpayer_id', how='left')
    else:
        compliance_agg[f'total_{col}' if col != 'compliance_rating' else f'avg_{col}'] = 0 if col != 'compliance_rating' else 70

features = features.merge(compliance_agg, on='taxpayer_id', how='left')
features['num_audits'].fillna(0, inplace=True)
features['avg_compliance_rating'].fillna(70, inplace=True)

# Feature 34-37
features['has_audit_issues'] = (features.get('num_audits_with_issues', 0) > 0).astype(int)
features['has_penalties'] = (features.get('total_penalties_imposed', 0) > 0).astype(int)
features['low_compliance_rating'] = (features['avg_compliance_rating'] < 60).astype(int)
features['never_audited'] = (features['num_audits'] == 0).astype(int)

print(f"✓ Created 4 compliance history features")


[STEP 7] Engineering compliance history features...
✓ Created 4 compliance history features


In [9]:
# ============================================================================
# STEP 8: PACRA VERIFICATION FEATURES
# ============================================================================
print("\n[STEP 8] Engineering PACRA verification features...")

features = features.merge(pacra, on='taxpayer_id', how='left')

# Feature 38-42
for col, feature_name, condition in [
    ('business_registered_pacra', 'not_registered_pacra', 'No'),
    ('directors_match', 'directors_mismatch', 'No'),
    ('registered_address_match', 'address_mismatch_pacra', 'No'),
    ('company_status', 'is_dormant_company', 'Dormant')
]:
    if col in features.columns:
        features[feature_name] = (features[col] == condition).astype(int)
    else:
        features[feature_name] = 0

if 'share_capital' in features.columns:
    features['share_capital'].fillna(0, inplace=True)
else:
    features['share_capital'] = 0

features['claim_exceeds_share_capital'] = (
    features['net_refund_amount'] > features['share_capital'] * 10
).astype(int)

print(f"✓ Created 5 PACRA verification features")


[STEP 8] Engineering PACRA verification features...
✓ Created 5 PACRA verification features


In [10]:
# ============================================================================
# STEP 9: COMPOSITE RISK SCORES
# ============================================================================
print("\n[STEP 9] Engineering composite risk scores...")

features['supplier_risk_score'] = (
    features['low_supplier_diversity'] * 3 +
    features['has_unverified_suppliers'] * 3 +
    features['has_unverified_payments'] * 2 +
    features['suspicious_purchase_pattern'] * 2
)

features['export_risk_score'] = (
    features['export_without_customs'] * 5 +
    features['has_missing_export_docs'] * 3 +
    features['has_unpaid_sales'] * 2
)

features['legitimacy_risk_score'] = (
    features['not_registered_pacra'] * 4 +
    features['address_not_verified'] * 2 +
    features['directors_mismatch'] * 2 +
    features['is_dormant_company'] * 2
)

features['historical_risk_score'] = (
    features['has_audit_issues'] * 3 +
    features['has_penalties'] * 3 +
    features['low_compliance_rating'] * 2 +
    features['is_first_time_claimant'] * 2
)

print(f"✓ Created 4 composite risk scores")


[STEP 9] Engineering composite risk scores...
✓ Created 4 composite risk scores


In [11]:

# ============================================================================
# STEP 10: MERGE LABELS AND PREPARE FINAL DATASET
# ============================================================================
print("\n[STEP 10] Merging labels and preparing final dataset...")

features = features.merge(labels[['claim_id', 'audit_result']], on='claim_id', how='left')
features['is_fraud'] = (features['audit_result'] == 'Fraud').astype(int)


[STEP 10] Merging labels and preparing final dataset...


In [12]:
# ============================================================================
# FINAL FEATURE SELECTION AND EXPORT
# ============================================================================
print("\n" + "="*70)
print("FEATURE ENGINEERING COMPLETE!")
print("="*70)

feature_columns = [
    'refund_to_output_ratio', 'input_output_vat_ratio', 'processing_days_zscore',
    'total_claims_filed', 'is_first_time_claimant', 'is_high_value_claim',
    'submitted_by_agent', 'has_few_documents',
    'business_age_years', 'is_young_business', 'address_not_verified',
    'bank_not_verified', 'is_small_business', 'claim_to_turnover_ratio',
    'claim_exceeds_business_size', 'single_director', 'no_tax_agent',
    'is_sole_proprietor',
    'low_supplier_diversity', 'has_unverified_suppliers', 
    'unverified_supplier_ratio', 'has_unverified_payments',
    'has_few_purchases', 'suspicious_purchase_pattern',
    'has_export_sales', 'is_export_heavy', 'has_missing_export_docs',
    'has_unpaid_sales', 'has_few_sales', 'single_export_destination',
    'export_without_customs', 'large_export_customs_mismatch',
    'has_unverified_customs',
    'has_audit_issues', 'has_penalties', 'low_compliance_rating',
    'never_audited',
    'not_registered_pacra', 'directors_mismatch', 'address_mismatch_pacra',
    'is_dormant_company', 'claim_exceeds_share_capital',
    'supplier_risk_score', 'export_risk_score', 'legitimacy_risk_score',
    'historical_risk_score'
]

final_features = features[['claim_id', 'taxpayer_id', 'net_refund_amount', 'is_fraud'] + feature_columns]

print(f"\n✓ Total Features Created: {len(feature_columns)}")
print(f"✓ Final Dataset Shape: {final_features.shape}")
print(f"✓ Fraud Cases: {final_features['is_fraud'].sum():,} ({final_features['is_fraud'].mean()*100:.1f}%)")
print(f"✓ Legitimate Cases: {(~final_features['is_fraud'].astype(bool)).sum():,}")

final_features.to_csv('zra_ml_features.csv', index=False)
print(f"\n✓ Saved to: zra_ml_features.csv")


print("\n" + "="*70)
print("Ready for Model Training! 🚀")
print("="*70)


FEATURE ENGINEERING COMPLETE!

✓ Total Features Created: 46
✓ Final Dataset Shape: (1000, 50)
✓ Fraud Cases: 150 (15.0%)
✓ Legitimate Cases: 850

✓ Saved to: zra_ml_features.csv

Ready for Model Training! 🚀
